# Record-cleaning: annotate F014 (notebook 10 construction artifact)

Run-once, **idempotent, append-only**. Notebook 10's auto-verdict (SHAP-ADDS-VALUE-ON-UNSW)
was a construction artifact; this marks F014 construction-limited and points to the corrected
result (F015 / notebook 10b). It does not overwrite F014 — the original run stands as the audit
trail, the annotation is the corrected interpretation. Re-running is a no-op (marker-guarded).

Writes:
1. **F014 correction** (findings_log) — the artifact diagnosis, what survives (dual adds
   nothing; modality-circularity; permutation-instability), and the 10b correction.
2. **Journal entry** — the diagnosis, the fix, and the drift-localization-vs-attack-attribution
   insight that the artifact exposes.

In [1]:
import datetime
from pathlib import Path
import sys
sys.path.insert(0, '/content/drive/MyDrive/phd_thesis')
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception:
    pass
from src.utils.documentation import add_journal_entry

ROOT = Path('/content/drive/MyDrive/phd_thesis')
NOTES = ROOT / 'reports' / 'advisor_notes'
FINDINGS = NOTES / 'findings_log.md'
JOURNAL = NOTES / 'thesis_journal.md'
TODAY = datetime.date.today().isoformat()

def append_if_absent(path, marker, block):
    path.parent.mkdir(parents=True, exist_ok=True)
    existing = path.read_text() if path.exists() else ''
    if marker in existing:
        print(f'[SKIP] {path.name}: "{marker[:42]}..." already present'); return False
    with open(path, 'a') as f:
        if existing and not existing.endswith('\n'):
            f.write('\n')
        f.write(block)
    print(f'[APPENDED] {path.name}: {marker[:42]}...'); return True

F014_MARK = '## F014 \u2014 CORRECTION (construction-limited; superseded by F015)'
F014_BLOCK = f'''
{F014_MARK}  ({TODAY})
**Applies to:** F014 (notebook 10, UNSW distributional benchmark). Annotation, not overwrite.

The auto-verdict **SHAP-ADDS-VALUE-ON-UNSW is a construction artifact** and does **not**
overturn C2. Two coupled flaws: (1) the AUC anchor (H-vs-benign separability) is the *same
task* as aux-SHAP (a benign-vs-H model's attribution), so the anchor was aligned with SHAP,
not neutral; (2) the leave-one-family-out construction put the other attacks in the pre-window,
so KS (which targets the pre->post input shift) was scored on the wrong contrast and fell
**below chance** (0.133 < 0.238 random). That measured attack-attribution, not
drift-localization -- and C2 is a drift-localization claim.

**What survives (robust to the flaw):** dual adds nothing (dual-aux = -0.100, GT-robust
internal-SHAP comparison); **modality-circularity** confirmed on UNSW (aux-SHAP inflates
+0.439 vs its own oracle-SHAP GT); **permutation-instability** confirmed (perm-dual 0.514 vs
aux-SHAP 0.729 self-stability).

**Correction:** notebook **10b (F015, D027)** re-tests C2 with an additive-drift construction
(post = baseline + injected H, baseline held constant), drift-localizers only (KS vs a
pre-vs-post domain-classifier SHAP vs the original dual-IDS disagreement), and a model-free
Wasserstein drift anchor. F015 supersedes the KS-vs-SHAP reading of F014; the dual /
modality / stability results above stand.
'''

append_if_absent(FINDINGS, F014_MARK, F014_BLOCK)

JTITLE = 'Notebook 10 (F014) construction artifact and the 10b correction'
JBODY = (
    'Notebook 10 returned SHAP-ADDS-VALUE-ON-UNSW, but this was a construction artifact, '
    'caught on review and corrected in 10b. Diagnosis: the AUC anchor (H-vs-benign) was the '
    'same task as aux-SHAP (a benign-vs-H model attribution), so it favoured SHAP; and the '
    'leave-one-family-out construction put the other attacks in the pre-window, so KS was '
    'scored on the pre-vs-post composition shift rather than H-vs-benign and fell below chance '
    '(0.133 < 0.238). The notebook tested attack-attribution, not drift-localization.\\n\\n'
    'What survives F014: dual adds nothing (dual-aux -0.100, GT-robust); modality-circularity '
    'confirmed on UNSW (aux-SHAP +0.439 vs oracle-SHAP); permutation-instability confirmed '
    '(perm 0.514 vs SHAP 0.729 self-stability).\\n\\n'
    'Fix (10b / F015 / D027): additive drift (post = baseline + injected H), drift-localizers '
    'only (KS vs pre-vs-post domain-classifier SHAP vs dual-IDS), and a model-free Wasserstein '
    'drift anchor (a different functional from KS). Validated on a toy: KS now sits above '
    'chance and competitive with domain-SHAP, and both recover known drifted features.\\n\\n'
    'Insight worth keeping for C1: the artifact is a category error -- explanation-of-the-IDS-'
    'model (attack-attribution) is a different target from distributional-drift-localization. '
    'The original dual-model hypothesis quietly conflated them. Naming and separating the two '
    'targets is itself a contribution of the evaluation framework. Run 10b, then the '
    'experimental programme is complete; remaining work is consolidation, the F001 P(X)-'
    'stability check, and the advisor conversation.'
)
if JOURNAL.exists() and JTITLE in JOURNAL.read_text():
    print(f'[SKIP] journal: entry "{JTITLE[:42]}..." already present')
else:
    add_journal_entry(title=JTITLE, body=JBODY)
    print('[JOURNAL ADDED]', JTITLE)

print('\nF014 annotation complete.')

Mounted at /content/drive
[APPENDED] findings_log.md: ## F014 — CORRECTION (construction-limited...
[JOURNAL ENTRY] Notebook 10 (F014) construction artifact and the 10b correction
[JOURNAL ADDED] Notebook 10 (F014) construction artifact and the 10b correction

F014 annotation complete.


## Done

Marker-guarded, so re-running prints `[SKIP]` for each write. The original F014 (notebook 10)
stands as the run record; this is the corrected interpretation. Run notebook 10b to produce
F015, the valid C2 test.